# Question 1: Taking out the reviews on the first page into a csv: "first_question.csv"

In [700]:
import urllib.request
import re

In [701]:
def request_page(url):
    headers = {
        'accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7',
        'accept-encoding': 'gzip, deflate, br, zstd',
        'accept-language': 'en-US,en;q=0.9,zh-CN;q=0.8,zh;q=0.7',
        'cache-control': 'max-age=0',
        'priority': 'u=0, i',
        'referer': 'https://www.google.com/',
        'sec-ch-ua': '"Not)A;Brand";v="99", "Google Chrome";v="127", "Chromium";v="127"',
        'sec-ch-ua-mobile': '?0',
        'sec-ch-ua-platform': '"macOS"',
        'sec-fetch-dest': 'document',
        'sec-fetch-mode': 'navigate',
        'sec-fetch-site': 'cross-site',
        'sec-fetch-user': '?1',
        'upgrade-insecure-requests': '1',
        'user-agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/139.0.0.0 Safari/537.36'
    }
    req = urllib.request.Request(url, headers=headers)
    response = urllib.request.urlopen(req)
    import gzip

    # Check if the response is compressed
    if response.info().get('Content-Encoding') == 'gzip':
        with gzip.GzipFile(fileobj=response) as gz:
            html = gz.read().decode('utf-8')
    else:
        html = response.read().decode('utf-8')
    return html

In [702]:
def safe_csv_field(s):
    s = str(s)
    if any(ch in s for ch in [',', '"', '\n', '\r']):
        return '"' + s.replace('"', '""') + '"'
    return s

In [703]:
def extract_total_reviews(html: str) -> int:

    div_match = re.search(r'<div class="results-summary".*?>(.*?)</div>', html, re.DOTALL)
    if not div_match:
        raise ValueError("Could not find a div with class 'results-summary'")

    div_content = div_match.group(1)

    # Now search for the "of <number> reviews" inside that div
    review_match = re.search(r'of\s+(\d+)\s+reviews', div_content)
    if review_match:
        return int(review_match.group(1))
    else:
        raise ValueError("Could not find total reviews in results-summary div.")

In [704]:
product_url = 'https://www.bestbuy.com/site/apple-iphone-14-pro-max-128gb-deep-purple-verizon/6487406.p?skuId=6487406'

In [705]:
start_idx = product_url.find('site')+5
sku = product_url[-7:]
sku_idx = product_url.find(sku)
search_url = product_url[:start_idx] + 'reviews/' + product_url[start_idx:sku_idx] + sku + '?variant=A&page=1'
links = []

In [706]:
html = request_page(search_url)

In [707]:
reviews_total = extract_total_reviews(html)
reviews_pages = reviews_total // 20
for page in range(1,reviews_pages+1):
    links.append(search_url[:-1] + str(page))

In [708]:
index = html.find("reviews-list")
html2 = html[index:]
review_content = []
reviewer = []
product_name = product_url[start_idx:sku_idx-1]

for review_counter in range(20):
    #We are running a while loop because we want to get everything in the page

    index = html2.find('ugc-author v-fw-medium body-copy-lg')
    remaining = html2[index:]
    index = html2.find('strong')
    remaining = html2[index:]
    start = remaining.find(">")
    end = remaining.find("<")
    reviewer.append(safe_csv_field(remaining[start+1:end]))

    remaining = remaining[end:]

    index =remaining.find('pre-white-space')
    remaining = remaining[index:]
    start = remaining.find(">")
    end = remaining.find("<")
    review_content.append(safe_csv_field(remaining[start+1:end]))

    remaining = remaining[end:]

    index =remaining.find('review-item')
    html2 = remaining[index:]

    review_counter += 1


In [709]:
Output_File = open("first_question.csv", "w")

Output_File.write("Product name,Review_username,Review_Content\n")

for x in range(20):
    Output_File.write(product_name+","+ reviewer[x] + "," + review_content[x] + "\n")

Output_File.close()

# Question 2: Taking another product and doing the same for it and saving the reviews in csv: "second_question.csv"

In [710]:
product_url = 'https://www.bestbuy.com/site/apple-iphone-16-pro-max-256gb-apple-intelligence-black-titanium-at-t/6443483.p?skuId=6443483'

In [711]:
start_idx = product_url.find('site')+5
sku = product_url[-7:]
sku_idx = product_url.find(sku)
search_url = product_url[:start_idx] + 'reviews/' + product_url[start_idx:sku_idx] + sku + '?variant=A&page=1'
links = []

In [712]:
html = request_page(search_url)

In [713]:
index = html.find("reviews-list")
html2 = html[index:]
review_content = []
reviewer = []
product_name = product_url[start_idx:sku_idx-1]

for review_counter in range(20):
    #We are running a while loop because we want to get everything in the page

    index = html2.find('ugc-author v-fw-medium body-copy-lg')
    remaining = html2[index:]
    index = html2.find('strong')
    remaining = html2[index:]
    start = remaining.find(">")
    end = remaining.find("<")
    reviewer.append(safe_csv_field(remaining[start+1:end]))

    remaining = remaining[end:]

    index =remaining.find('pre-white-space')
    remaining = remaining[index:]
    start = remaining.find(">")
    end = remaining.find("<")
    review_content.append(safe_csv_field(remaining[start+1:end]))

    remaining = remaining[end:]

    index =remaining.find('review-item')
    html2 = remaining[index:]

    review_counter += 1

In [714]:
Output_File = open("second_question.csv", "w")

Output_File.write("Product name,Review_username,Review_Content\n")

for x in range(20):
    Output_File.write(product_name+","+ reviewer[x] + "," + review_content[x] + "\n")

Output_File.close()